# AfroXLMR multilingual domain classifier

One model for Swahili, Kinyarwanda and Nigerian Pidgin. Labels: agriculture, finance, health, law, telecoms (one or two per text) or other.

**Before running:** Settings → Accelerator → **GPU T4**; Settings → **Internet on** (to download `Davlan/afro-xlmr-base`); Add Input → your private dataset with the training files. The dataset contains corpus text: keep it private.

In [ ]:
import glob, os, torch, transformers
hits = glob.glob('/kaggle/input/**/train_afroxlmr_domain.py', recursive=True)
assert hits, 'Add the private training dataset to this notebook (Add Input > Your Datasets)'
SRC = os.path.dirname(hits[0]); OUT = '/kaggle/working/afroxlmr_out'
print(SRC, sorted(os.listdir(SRC)))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: Settings > Accelerator > GPU T4')
print('torch', torch.__version__, 'transformers', transformers.__version__)

## 1. Quick check (about 3 minutes)
A few hundred rows, one epoch: confirms the model downloads and the code runs on this GPU. The scores here mean nothing.

In [ ]:
!python {SRC}/train_afroxlmr_domain.py --data-dir {SRC} --out-dir /kaggle/working/smoke --max-train 400 --max-eval 200 --epochs 1

## 2. Full training (about 30 to 60 minutes on a T4)
45,685 training rows (at most 4,000 per language and label), 3 epochs; the best epoch on dev is kept.

In [ ]:
!python {SRC}/train_afroxlmr_domain.py --data-dir {SRC} --out-dir {OUT} --epochs 3

## 3. Test results (human-labelled test sets)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f'{OUT}/report.md', encoding='utf-8').read()))

## 4. Classify sample texts from each language
Held-out test texts (never trained on): the model's labels next to the human labels.

In [ ]:
!python {SRC}/predict_domain_afroxlmr.py {SRC}/demo_samples.csv /kaggle/working/demo_predictions.csv --model-dir {OUT}/model --text-col text --lang-col language
import pandas as pd
pd.set_option('display.max_colwidth', 100)
d = pd.read_csv('/kaggle/working/demo_predictions.csv')
display(d[['language', 'human_labels', 'domains', 'text']])

## 5. Package the results
`afroxlmr_results.zip` (small) has the report, metrics, predictions and training log. The model itself stays in `afroxlmr_out/model` in this notebook's Output.

In [ ]:
!rm -rf /kaggle/working/smoke
!cd /kaggle/working && zip -qr afroxlmr_results.zip afroxlmr_out/report.md afroxlmr_out/metrics.csv afroxlmr_out/test_predictions.csv afroxlmr_out/train_log.json afroxlmr_out/model/thresholds.json demo_predictions.csv && ls -la /kaggle/working /kaggle/working/afroxlmr_out/model